# 05 - Bronze: ACLED local fallback ingestion

Databricks Free Edition serverless cannot resolve `acleddata.com` from this workspace. ACLED extraction therefore runs locally on the Mac, writes JSONL, and this notebook ingests the uploaded JSONL from a Databricks Volume.

Local extraction:

```bash
python3 extraction/extract/acled_extract.py \
  --all-cemac-ecowas \
  --start-year 2010 --end-year 2026 \
  --out data/raw/acled/cemac_ecowas_acled_2010_2026.jsonl
```

The script reads `ACLED_USERNAME` and `ACLED_PASSWORD` from the environment or `.env`; if missing, it prompts interactively so the password does not go into shell history.

Upload to Databricks:

```bash
databricks fs mkdirs dbfs:/Volumes/cemac_ecowas_aes_trade/bronze/raw_landing/acled -p cemac-project
databricks fs cp data/raw/acled/cemac_ecowas_acled_2010_2026.jsonl \
  dbfs:/Volumes/cemac_ecowas_aes_trade/bronze/raw_landing/acled/cemac_ecowas_acled_2010_2026.jsonl \
  --overwrite -p cemac-project
```

Outputs:

- `bronze.acled_events_historical`
- `bronze.acled_weekly_aggregated`

In [ ]:
# Cell 1 - Configuration
from datetime import datetime, timezone

from pyspark.sql import functions as F

spark.sql("USE CATALOG cemac_ecowas_aes_trade")
spark.sql("USE SCHEMA bronze")

VOLUME_PATH = "/Volumes/cemac_ecowas_aes_trade/bronze/raw_landing/acled/"
loaded_at = datetime.now(timezone.utc)

print("Catalog: cemac_ecowas_aes_trade")
print("Schema: bronze")
print(f"Volume path: {VOLUME_PATH}")

In [ ]:
# Cell 2 - List uploaded JSONL files
files = [f for f in dbutils.fs.ls(VOLUME_PATH) if f.name.endswith(".jsonl")]

if not files:
    raise FileNotFoundError(
        f"No .jsonl files found at {VOLUME_PATH}. Run acled_extract.py locally and upload the output first."
    )

print(f"Found {len(files)} JSONL file(s):")
for file in files:
    print(f"  {file.name} ({file.size:,} bytes)")

In [ ]:
# Cell 3 - Read JSONL envelopes and explode event payloads
paths = [file.path for file in files]
envelopes_df = spark.read.json(paths)

print(f"Envelope rows: {envelopes_df.count():,}")
envelopes_df.select(
    "reporter_iso3", "reporter_name", "bloc_seed", "query_year",
    F.size("payload").alias("payload_rows"),
).orderBy("reporter_iso3", "query_year").show(50, truncate=False)

exploded_df = envelopes_df.withColumn("event", F.explode_outer("payload"))

In [ ]:
# Cell 4 - Build event-level Spark DataFrame
events_df_raw = exploded_df.where(F.col("event").isNotNull()).select(
    F.col("event.event_id_cnty").cast("string").alias("event_id_cnty"),
    F.col("event.event_date").cast("string").alias("event_date"),
    F.col("event.year").cast("int").alias("year"),
    F.col("event.time_precision").cast("int").alias("time_precision"),
    F.col("event.disorder_type").cast("string").alias("disorder_type"),
    F.col("event.event_type").cast("string").alias("event_type"),
    F.col("event.sub_event_type").cast("string").alias("sub_event_type"),
    F.col("event.actor1").cast("string").alias("actor1"),
    F.col("event.assoc_actor_1").cast("string").alias("assoc_actor_1"),
    F.expr("try_cast(event.inter1 as int)").alias("inter1"),
    F.col("event.actor2").cast("string").alias("actor2"),
    F.col("event.assoc_actor_2").cast("string").alias("assoc_actor_2"),
    F.expr("try_cast(event.inter2 as int)").alias("inter2"),
    F.expr("try_cast(event.interaction as int)").alias("interaction"),
    F.col("event.civilian_targeting").cast("string").alias("civilian_targeting"),
    F.col("event.iso").cast("string").alias("iso"),
    F.col("reporter_iso3").cast("string").alias("iso3"),
    F.col("event.country").cast("string").alias("country"),
    F.col("reporter_name").cast("string").alias("project_country_name"),
    F.col("bloc_seed").cast("string").alias("bloc_seed"),
    F.col("event.region").cast("string").alias("region"),
    F.col("event.admin1").cast("string").alias("admin1"),
    F.col("event.admin2").cast("string").alias("admin2"),
    F.col("event.admin3").cast("string").alias("admin3"),
    F.col("event.location").cast("string").alias("location"),
    F.col("event.latitude").cast("double").alias("latitude"),
    F.col("event.longitude").cast("double").alias("longitude"),
    F.col("event.geo_precision").cast("int").alias("geo_precision"),
    F.col("event.source").cast("string").alias("source_name"),
    F.col("event.source_scale").cast("string").alias("source_scale"),
    F.expr("try_cast(event.fatalities as int)").alias("fatalities"),
    F.col("event.tags").cast("string").alias("tags"),
    F.col("event.timestamp").cast("long").alias("acled_timestamp"),
    F.col("extracted_at").cast("string").alias("extracted_at"),
)

events_df = (
    events_df_raw
    .dropDuplicates(["event_id_cnty"])
    .withColumn("event_date_dt", F.to_date("event_date"))
    .withColumn("source", F.lit("acled"))
    .withColumn("loaded_at", F.lit(loaded_at))
)

print(f"Raw event rows: {events_df_raw.count():,}")
print(f"Deduped event rows: {events_df.count():,}")
events_df.printSchema()
events_df.show(10, truncate=False)

In [ ]:
# Cell 5 - Build weekly aggregate DataFrame
weekly_df = (
    events_df
    .withColumn(
        "week_start_date",
        F.expr("date_sub(event_date_dt, pmod(dayofweek(event_date_dt) + 5, 7))")
    )
    .groupBy(
        "iso3", "country", "project_country_name", "bloc_seed",
        "year", "week_start_date", "admin1", "event_type", "sub_event_type",
    )
    .agg(
        F.count("*").alias("event_count"),
        F.sum(F.coalesce(F.col("fatalities"), F.lit(0))).alias("fatalities"),
        F.countDistinct("event_id_cnty").alias("distinct_events"),
    )
    .withColumn("source", F.lit("acled"))
    .withColumn("loaded_at", F.lit(loaded_at))
)

print(f"Weekly aggregate rows: {weekly_df.count():,}")
weekly_df.show(10, truncate=False)

In [ ]:
# Cell 6 - Write bronze tables
spark.sql("DROP TABLE IF EXISTS bronze.acled_events_historical")
spark.sql("DROP TABLE IF EXISTS bronze.acled_weekly_aggregated")

(events_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("bronze.acled_events_historical"))

(weekly_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("bronze.acled_weekly_aggregated"))

print("Write complete:")
print("  bronze.acled_events_historical")
print("  bronze.acled_weekly_aggregated")

In [ ]:
# Cell 7 - Verification and sanity checks
print("Event-level coverage:")
spark.sql("""
    SELECT
        COUNT(*) AS total_events,
        COUNT(DISTINCT iso3) AS countries,
        MIN(event_date_dt) AS earliest_event_date,
        MAX(event_date_dt) AS latest_event_date,
        SUM(fatalities) AS total_fatalities
    FROM bronze.acled_events_historical
""").show()

print("Per-country event coverage:")
spark.sql("""
    SELECT
        iso3,
        project_country_name,
        COUNT(*) AS events,
        COUNT(DISTINCT year) AS years_present,
        SUM(fatalities) AS fatalities
    FROM bronze.acled_events_historical
    GROUP BY iso3, project_country_name
    ORDER BY events DESC
""").show(25, truncate=False)

print("Weekly aggregate coverage:")
spark.sql("""
    SELECT
        COUNT(*) AS weekly_rows,
        COUNT(DISTINCT iso3) AS countries,
        MIN(week_start_date) AS earliest_week,
        MAX(week_start_date) AS latest_week,
        SUM(event_count) AS total_events_reconstructed,
        SUM(fatalities) AS total_fatalities_reconstructed
    FROM bronze.acled_weekly_aggregated
""").show()

print("Top event types since 2010:")
spark.sql("""
    SELECT
        event_type,
        COUNT(*) AS events,
        SUM(fatalities) AS fatalities
    FROM bronze.acled_events_historical
    GROUP BY event_type
    ORDER BY events DESC
""").show(truncate=False)

print("Cameroon admin1 conflict intensity, latest 3 years:")
spark.sql("""
    SELECT
        admin1,
        COUNT(*) AS events,
        SUM(fatalities) AS fatalities
    FROM bronze.acled_events_historical
    WHERE iso3 = 'CMR'
      AND year >= YEAR(CURRENT_DATE()) - 3
    GROUP BY admin1
    ORDER BY events DESC
    LIMIT 10
""").show(truncate=False)